In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 25.7 MB/s eta 0:00:00
ERROR: Operation cancelled by user


KeyboardInterrupt: 

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO

# =====================================================
# 1. 경로 설정
# =====================================================
video_folder = r"/content/drive/MyDrive/Colab Notebooks/포오올더/mlb_mediafile"
output_folder = r"/content/drive/MyDrive/Colab Notebooks/포오올더"

os.makedirs(output_folder, exist_ok=True)

model_path = r"/content/drive/MyDrive/Colab Notebooks/포오올더/best (1).pt"
model = YOLO(model_path)

# =====================================================
# 2. 영상 리스트 가져오기
# =====================================================
video_files = [f for f in os.listdir(video_folder) if f.endswith(".mp4")]

print(f"총 영상 개수: {len(video_files)}")

# =====================================================
# 3. 각 영상별 공 좌표 추출
# =====================================================
for video_name in video_files:

    video_path = os.path.join(video_folder, video_name)
    cap = cv2.VideoCapture(video_path)

    ball_positions = []
    frame_idx = 0

    print(f"처리 중: {video_name}")

    while True:
        ret, frame = cap.read()
        if not ret:
            break

        results = model(frame, verbose=False)
        boxes = results[0].boxes

        if boxes is not None and len(boxes) > 0:
            box = boxes.xyxy[0].cpu().numpy()

            x_center = (box[0] + box[2]) / 2
            y_center = (box[1] + box[3]) / 2

            ball_positions.append((frame_idx, x_center, y_center))
        else:
            ball_positions.append((frame_idx, np.nan, np.nan))

        frame_idx += 1

    cap.release()

    # =====================================================
    # 4. CSV 저장 (영상 이름 그대로)
    # =====================================================
    output_csv_name = video_name.replace(".mp4", ".csv")
    output_csv_path = os.path.join(output_folder, output_csv_name)

    ball_df = pd.DataFrame(ball_positions, columns=["video_frame", "ball_x", "ball_y"])
    ball_df.to_csv(output_csv_path, index=False)

    print(f"저장 완료: {output_csv_path}")

print("🎯 전체 영상 공 좌표 추출 완료")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')